# Chapter 13 — Verification, Repair, and Rejection

**Book alignment:** Hallucination From First Principles, Chapter 13

**Question this notebook isolates:** Does a bounded recovery loop terminate with PERMIT when repair resolves the evidence gap, ABSTAIN when rhetoric changes without evidence change, and reject unknown repair modes before running?

Synthetic fixtures with a perfect structured oracle demonstrate control invariants, not real verifier accuracy.


In [ ]:
from pathlib import Path
import sys

import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "hallucination-from-first-principles").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/hallucination-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "hallucination-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import recovery_demo as rec
from recovery_demo import RecoveryController, base_candidate, verify, signature, V
import policy_engine as pe

rng = np.random.default_rng(13)


## Omission repair terminates PERMIT

The verifier marks `c3` (Q3 revenue) `INSUFFICIENT_EVIDENCE` because the evidence snapshot holds Q2 revenue and the outlook but no Q3 value. Omission deletes the unsupported claim, producing a new candidate whose every claim verifies.


In [ ]:
terminal_omit, final_omit = RecoveryController().run(base_candidate(), "omission")
states_omit = verify(final_omit, rec.EVIDENCE)

print("terminal:", terminal_omit)
print("final:", final_omit.id, "parent:", final_omit.parent)
print("claims:", [(c.id, c.text) for c in final_omit.claims])
print("states:", {k: v.value for k, v in states_omit.items()})


In [ ]:
assert terminal_omit == "PERMIT"
assert final_omit.authorized is True
assert [c.id for c in final_omit.claims] == ["c1", "c2"]
assert all(v == V.SUPPORTED for v in states_omit.values())
assert final_omit.parent == "cand_v1"

print("omission: unsupported claim removed, new state authorized")


## Hedging without new evidence terminates ABSTAIN:CYCLE_DETECTED

The hedge operator softens wording (`was` to `may have been`) without adding the missing Q3 value. Verification state never changes, the candidate signature repeats, and cycle detection stops the loop instead of trusting the rewrite.


In [ ]:
terminal_hedge, final_hedge = RecoveryController().run(base_candidate(), "hedge")
states_hedge = verify(final_hedge, rec.EVIDENCE)

print("terminal:", terminal_hedge)
print("final:", final_hedge.id, "parent:", final_hedge.parent)
print("c3 text:", [c.text for c in final_hedge.claims if c.id == "c3"])
print("states:", {k: v.value for k, v in states_hedge.items()})


In [ ]:
assert terminal_hedge == "ABSTAIN:CYCLE_DETECTED"
assert [c.id for c in final_hedge.claims] == ["c1", "c2", "c3"]
assert states_hedge["c3"] == V.INSUFFICIENT_EVIDENCE

print("hedge: rhetoric changed, evidence state did not -> abstain")


## Unknown mode raises; repetition and regression are measured

An undeclared repair mode is rejected before any repair runs. The second hedge is idempotent, so the `(candidate hash, verification state)` signature repeats exactly. A careless patch that fixes one property while breaking another is scored with candidate-level repair-induced regression (RSI).


In [ ]:
try:
    RecoveryController().run(base_candidate(), "guess")
    raised = None
except ValueError as exc:
    raised = str(exc)
print("raised:", raised)

c0 = base_candidate()
s0 = verify(c0, rec.EVIDENCE)
c1 = rec.repair_hedge(c0, s0, "cand_v2")
s1 = verify(c1, rec.EVIDENCE)
c2 = rec.repair_hedge(c1, s1, "cand_v3")
s2 = verify(c2, rec.EVIDENCE)
print("sig(cand_v2) == sig(cand_v3):", signature(c1, s1) == signature(c2, s2))

# Whac-A-Mole demo: date fixed, relation flipped (object regressed)
props_before = {"subject", "object", "polarity"}
props_after = {"subject", "polarity", "date"}
rsi = len(props_before - props_after) / max(1, len(props_before))
print("RSI:", round(rsi, 4))


In [ ]:
assert raised is not None and "unknown repair mode" in raised
assert signature(c1, s1) == signature(c2, s2)
assert abs(rsi - 1 / 3) < 1e-12

print("unknown mode rejected; repeated signature proves stall; RSI exposes regression")


## What we earned

A repaired claim is still a claim: omission earns `PERMIT` only as a new, fully supported candidate state, while hedging the same gap without evidence earns `ABSTAIN:CYCLE_DETECTED`. Unknown repairs are refused up front, repeated verification signatures prove a stalled strategy, and RSI names Whac-A-Mole regressions.

Notebook 14 / Chapter 14 keeps this bounded loop and adds the persistence boundary: repair candidates stay ephemeral until an explicit admission policy grants durable capabilities.
